## O código a seguir contem a implementação de um Sistema de Gestão de Estudantes usando Pydantic

### Modelagem, Validação Avançada e Serialização

In [4]:
!pip install 'pydantic[email]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 8.9 MB/s eta 0:00:00


In [5]:
import enum
import hashlib
import re
from typing import Any
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_serializer,
    field_validator,
    model_serializer,
    model_validator,
    SecretStr,
    ValidationError
)

# Matrícula: Deve começar com '20' seguido de 6 dígitos (ex: 20240123)
REGEX_MATRICULA_VALIDA = re.compile(r"^20\d{6}$")
# Nome: Apenas letras e espaços, mínimo de 3 caracteres
REGEX_NOME_VALIDO = re.compile(r"^[a-zA-ZÀ-ÿ\s]{3,}$")

class StatusAcademico(enum.IntFlag):
    Matriculado = 1
    Suspenso = 2
    Formado = 4
    Evadido = 8

class Aluno(BaseModel):
    nome: str = Field(examples=["Ada Lovelace"])

    matricula: str = Field(
        examples=["20250001"],
        description="Número de matrícula único do estudante"
    )

    email: EmailStr = Field(
        examples=["ada@ufg.br"],
        frozen=True, # Imutável após a criação da instância
    )

    senha_acesso: SecretStr = Field(
        description="Senha de acesso ao portal do aluno",
        exclude=True # NUNCA aparecerá no dump ou JSON gerado
    )

    status: StatusAcademico = Field(
        default=StatusAcademico.Matriculado,
        validate_default=True
    )

    @field_validator("nome")
    @classmethod
    def validar_nome(cls, v: str) -> str:
        if not REGEX_NOME_VALIDO.match(v):
            raise ValueError("O nome deve conter apenas letras e ter no mínimo 3 caracteres.")
        return v.title() # Capitaliza as primeiras letras do nome automaticamente

    @field_validator("matricula")
    @classmethod
    def validar_matricula(cls, v: str) -> str:
        if not REGEX_MATRICULA_VALIDA.match(v):
            raise ValueError("A matrícula deve começar com '20' seguido de 6 dígitos numéricos.")
        return v

    # Converte strings ou inteiros para o Enum StatusAcademico
    @field_validator("status", mode="before")
    @classmethod
    def validar_status(cls, v: int | str | StatusAcademico) -> StatusAcademico:
        op = {int: lambda x: StatusAcademico(x), str: lambda x: StatusAcademico[x], StatusAcademico: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            opcoes = ', '.join([x.name for x in StatusAcademico])
            raise ValueError(f"Status inválido. Opções aceitas: {opcoes}")

    # Validação no nível do modelo (pode acessar e comparar vários campos)
    @model_validator(mode="before")
    @classmethod
    def validar_aluno_pre(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "senha_acesso" in v and "nome" in v:
            primeiro_nome = v["nome"].split()[0].casefold()
            if primeiro_nome in v["senha_acesso"].casefold():
                raise ValueError("A senha de acesso não pode conter o primeiro nome do aluno por segurança.")

            # Aplica o Hash na senha antes mesmo do objeto ser instanciado
            v["senha_acesso"] = hashlib.sha256(v["senha_acesso"].encode()).hexdigest()
        return v

    @field_serializer("status", when_used="json")
    def serializar_status(self, v: StatusAcademico) -> str:
        return v.name

    # Controla o formato final do JSON do modelo inteiro
    @model_serializer(mode="wrap", when_used="json")
    def serializar_aluno(self, serializador_padrao, info) -> dict[str, Any]:
        if not info.include and not info.exclude:
            return {
                "matricula": self.matricula,
                "nome": self.nome,
                "status": self.status.name
            }
        return serializador_padrao(self)


def main():
    try:
        # Testando a criação de um aluno válido
        aluno_valido = Aluno(
            nome="Alan Turing",
            matricula="20240456",
            email="alan@universidade.br",
            senha_acesso="SenhaForte123",
            status="Formado"
        )
        print("--- Aluno Válido ---")
        print("Print normal do Python:", aluno_valido)
        print("Saída JSON customizada:", aluno_valido.model_dump(mode="json"))

    except ValidationError as erro:
        print("Erro de validação:", erro)

if __name__ == "__main__":
    main()

--- Aluno Válido ---
Print normal do Python: nome='Alan Turing' matricula='20240456' email='alan@universidade.br' senha_acesso=SecretStr('**********') status=<StatusAcademico.Formado: 4>
Saída JSON customizada: {'matricula': '20240456', 'nome': 'Alan Turing', 'status': 'Formado'}


### Integrando com FastAPI

In [7]:
from datetime import datetime
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_serializer, UUID4

app = FastAPI()

class RegistroAluno(BaseModel):
    model_config = {"extra": "forbid"}

    __alunos__ = [] # Lista simulando um banco de dados

    id: UUID4 = Field(default_factory=uuid4, kw_only=True)
    nome: str = Field(..., description="Nome completo do estudante")
    email: EmailStr = Field(...)

    disciplinas_matriculadas: list[str] = Field(
        default_factory=list,
        max_length=10,
        description="Lista de disciplinas nas quais o aluno está matriculado"
    )

    data_registro: datetime = Field(
        default_factory=datetime.now, kw_only=True
    )

    # Converte o UUID para string no JSON final
    @field_serializer("id", when_used="json")
    def serializar_id(self, id: UUID4) -> str:
        return str(id)

    # Formata a data de registro para um padrão mais legível
    @field_serializer("data_registro", when_used="json")
    def serializar_data(self, dt: datetime) -> str:
        return dt.strftime("%d/%m/%Y %H:%M:%S")

@app.get("/alunos", response_model=list[RegistroAluno])
async def listar_alunos():
    return list(RegistroAluno.__alunos__)

@app.post("/alunos", response_model=RegistroAluno)
async def registrar_aluno(aluno: RegistroAluno):
    RegistroAluno.__alunos__.append(aluno)
    return aluno

@app.get("/alunos/{id_aluno}", response_model=RegistroAluno)
async def buscar_aluno_por_id(id_aluno: UUID4):
    try:
        return next((a for a in RegistroAluno.__alunos__ if a.id == id_aluno))
    except StopIteration:
        return JSONResponse(status_code=404, content={"mensagem": "Aluno não encontrado no sistema"})

def executar_testes() -> None:
    with TestClient(app) as cliente:
        # 1. Cadastro em lote
        for i in range(3):
            resposta = cliente.post(
                "/alunos",
                json={
                    "nome": f"Estudante {i}",
                    "email": f"estudante{i}@universidade.br",
                    "disciplinas_matriculadas": ["Engenharia de Software", "Inteligência Artificial"]
                },
            )
            assert resposta.status_code == 200

        # 2. Verifica se a listagem retorna os 3 cadastrados
        resposta = cliente.get("/alunos")
        assert len(resposta.json()) == 3

        # 3. Busca específica pelo ID do primeiro aluno retornado
        id_primeiro_aluno = resposta.json()[0]["id"]
        resposta_individual = cliente.get(f"/alunos/{id_primeiro_aluno}")
        assert resposta_individual.status_code == 200
        assert resposta_individual.json()["nome"] == "Estudante 0"

        # 4. Testa a trava do 'extra: forbid' enviando um payload sujo
        resposta_invalida = cliente.post(
            "/alunos",
            json={
                "nome": "Hacker",
                "email": "hacker@malicioso.com",
                "tentativa_injetar_campo": "admin"
            }
        )
        assert resposta_invalida.status_code == 422

    print("Todos os testes da API passaram! Validações e bloqueios funcionando corretamente.")

if __name__ == "__main__":
    executar_testes()

Todos os testes da API passaram! Validações e bloqueios funcionando corretamente.
